### Import the necessary database

In [ ]:
import numpy as np
import xarray as xr
import pandas as pd
import matplotlib.pyplot as plt
import cartopy.crs as ccrs
import cartopy.feature as cfeature
import os

In [ ]:
#In[2]:
# define function
import src.SAT_function_Obs_Fingerprint as data_process
import src.Data_Preprocess as preprosess

In [ ]:
# import src.slurm_cluster as scluster
# client, scluster = scluster.init_dask_slurm_cluster(scale=4, cores=50, memory="200GB")

In [ ]:
# read all segmented trend patterns (L=10..73) into a dict of DataArrays
dir_ICV_seg_TREND = '/work/mh0033/m301036/OBS_LPS_revision/docs/data/FIG3/OBS_ICV_std/'
len_segments = np.arange(10, 74, 1)

ds = {}
for L in len_segments:
    fpath = os.path.join(dir_ICV_seg_TREND, f"OBS_ICV_MK_trend_segments_L{L}.nc")
    da = xr.open_dataset(fpath, chunks={"lat": 10, "lon": 10})['trend']
    ds[f"ICV_trend_{L}yr"] = da


In [ ]:
ds

### Calculate the regional mean of each trend Length 

In [ ]:
# flip the longitude to -180 to 180
for key in ds.keys():
    ds[key] = preprosess.convert_longitude(ds[key])

In [ ]:
lat = ds["ICV_trend_10yr"].lat
lon = ds["ICV_trend_10yr"].lon

In [ ]:
# Define the trapezoid region
# Southeast Pacific coordinates
lat5 = -25
lat6 = 0
lon5 = -160
lon6 = -80
lon7 = -110

In [ ]:
ds["ICV_trend_10yr"]

In [ ]:
def plot_trend(temp_data, lats, lons, levels=None, extend=None, cmap=None, 
                                 title="", ax=None, show_xticks=False, show_yticks=False):
    """
    Plot the trend spatial pattern using Robinson projection with significance overlaid.

    Parameters:
    - temp_data: 2D numpy array with the trend values.
    - lats, lons: 1D arrays of latitudes and longitudes.
    - p_values: 2D array with p-values for each grid point.
    - GMST_p_values: 2D array with GMST p-values for each grid point.
    - title: Title for the plot.
    - ax: Existing axis to plot on. If None, a new axis will be created.
    - show_xticks, show_yticks: Boolean flags to show x and y axis ticks.
    
    Returns:
    - contour_obj: The contour object from the plot.
    """
    # Plotting
    contour_obj = ax.contourf(lons, lats, temp_data, levels=levels, extend=extend, cmap=cmap, transform=ccrs.PlateCarree())

    ax.coastlines(resolution='110m')
    gl = ax.gridlines(draw_labels=True, dms=True, x_inline=False, y_inline=False,
                      color='gray', alpha=0.35, linestyle='--')

    # Disable labels on the top and right of the plot
    gl.top_labels = False
    gl.right_labels = False

    # Enable labels on the bottom and left of the plot
    gl.bottom_labels = show_xticks
    gl.left_labels = show_yticks
    gl.xformatter = cticker.LongitudeFormatter()
    gl.yformatter = cticker.LatitudeFormatter()
    gl.xlabel_style = {'size': 16}
    gl.ylabel_style = {'size': 16}
    
    if show_xticks:
        gl.bottom_labels = True
    if show_yticks:
        gl.left_labels = True
    
    # ax.set_title(title, loc='center', fontsize=18, pad=5.0)

    return contour_obj
# %%
plt.rcParams['figure.figsize'] = (8, 10)
plt.rcParams['font.size'] = 16
# plt.rcParams['font.family'] = 'sans-serif'
plt.rcParams['axes.labelsize'] = 16
plt.rcParams['ytick.direction'] = 'out'
plt.rcParams['ytick.minor.visible'] = True
plt.rcParams['ytick.major.right'] = True
plt.rcParams['ytick.right'] = True
plt.rcParams['xtick.bottom'] = True
plt.rcParams['savefig.dpi'] = 300
plt.rcParams['savefig.bbox'] = 'tight'
plt.rcParams['savefig.pad_inches'] = 0.1
plt.rcParams['savefig.transparent'] = True

import cartopy.crs as ccrs
import matplotlib.pyplot as plt
import matplotlib.colors as colors
import matplotlib.ticker as mticker
import cartopy.feature as cfeature
import cartopy.mpl.ticker as cticker
import matplotlib.patches as mpatches
import matplotlib.lines as mlines
import matplotlib.gridspec as gridspec
import matplotlib as mpl
import seaborn as sns
from matplotlib.colors import ListedColormap
from matplotlib.colors import BoundaryNorm, ListedColormap
import cartopy.util as cutil
import seaborn as sns
import matplotlib.colors as mcolors
import palettable

In [ ]:
da_plot =ds["ICV_trend_10yr"].sel(segment=slice(1,10)).mean(dim='segment')

fig, ax = plt.subplots(subplot_kw={'projection': ccrs.Robinson()})

contour_obj =  plot_trend(da_plot, da_plot.lat, da_plot.lon, levels=np.arange(-0.5, 0.6, 0.1), extend='both', cmap='RdBu_r',
                                    title="SEP region SAT anomaly 2020", ax=ax, show_xticks=True, show_yticks=True)

# colorbar
cbar = plt.colorbar(contour_obj, ax=ax, orientation='horizontal', pad=0.05, aspect=50)
cbar.set_label('Temperature anomaly (°C)')
cbar.ax.tick_params(labelsize=14)

plt.show()

In [ ]:
ds_sel_sep = {}
for key in ds.keys():
    ds_sel_sep[key] = ds[key].sel(lat=slice(lat5, lat6), lon=slice(lon5, lon6))

In [ ]:
ds_sel_sep

In [ ]:
import src.Polygon_region as poly
# %%
# define the polygon region 

if __name__ == '__main__':
    import matplotlib.pyplot as pl
    pl.close('all')

    # Dummy data.
    lats = ds_sel_sep['ICV_trend_10yr'].lat.values
    lons = ds_sel_sep['ICV_trend_10yr'].lon.values
    print(lats)
    print(lons)
    
    data = np.arange(lats.size*lons.size).reshape((lats.size, lons.size))
    
    # Bounding box.
    poly_x = np.array([-110, -160, -80, -80, -110])  # Longitude values
    poly_y = np.array([-25, 0, 0, -25, -25])         # Latitude values
    
    # Generate mask for calculating statistics.
    mask = np.zeros_like(data, dtype=bool)
    poly.get_mask(mask, lons, lats, poly_x, poly_y)
    
    # Calculate statistics.
    max_val = data[mask].max()
    
    # Plot data and mask.
    pl.figure(figsize=(10,4))
    pl.subplot(121)
    pl.title('data')
    pl.pcolormesh(lons, lats, data)
    pl.plot(poly_x, poly_y)
    pl.colorbar()
    
    pl.subplot(122)
    pl.title('averaging mask, max_value={}'.format(max_val))
    pl.pcolormesh(lons, lats, mask)
    pl.plot(poly_x, poly_y)
    pl.colorbar()
    
    pl.tight_layout()

In [ ]:
mask_da = xr.DataArray(mask, coords={"lat": ds_sel_sep['ICV_trend_10yr'].lat, "lon": ds_sel_sep['ICV_trend_10yr'].lon}, dims=["lat", "lon"])
mask_da

In [ ]:
ds_sel_sep_masked = {}
for key in ds_sel_sep.keys():
    ds_sel_sep_masked[key] = ds_sel_sep[key].where(mask_da)

In [ ]:
ds_sel_sep_masked

### specify the 5 and 95 percentile values for each year step and output the 10-73yr unforced percentile timeseires for each regions

In [ ]:
# define the function to calculate the percentile
def calc_percentile(da, q):
    """ Calculate the qth percentile of the data along the specified dimension.
    Args:
    da: xr.DataArray
    dim: str
    q: float
    Returns:
    xr.DataArray
    """
    # remove nans for da
    da = da.dropna(dim='segment')
    lower_percentile = np.percentile(da, q)
    upper_percentile = np.percentile(da, 100-q)
    
    return lower_percentile, upper_percentile

In [ ]:
ds_sel_sep_masked['ICV_trend_11yr']

In [ ]:
# calculate the regional mean anomalies of trend patterns
ds_sel_sep_masked_mean = {}
for key in ds_sel_sep_masked.keys():
    ds_sel_sep_masked_mean[key] = ds_sel_sep_masked[key].mean(dim=['lat', 'lon'], skipna=True)

In [ ]:
ds_sel_sep_masked_mean

In [ ]:
# calculate the regional mean's percentile
# 5%---[0]
unforced_trend_SEP_lower_percentile = {}

# 95%---[1]
unforced_trend_SEP_upper_percentile = {}

for key in ds_sel_sep_masked.keys():
    unforced_trend_SEP_lower_percentile[key], unforced_trend_SEP_upper_percentile[key] = calc_percentile(ds_sel_sep_masked_mean[key], 5)
    

In [ ]:
unforced_trend_SEP_lower_percentile.keys()

In [ ]:
list(unforced_trend_SEP_lower_percentile.values())

In [ ]:
# transform the dictionary to the dataarray
icv_years = list(unforced_trend_SEP_lower_percentile.keys())
unforced_trend_SEP_lower_percentile_da = xr.DataArray(
	list(unforced_trend_SEP_lower_percentile.values()),
	coords={'ICV_trend_year_lower': icv_years},
	dims=['ICV_trend_year_lower'],
)
unforced_trend_SEP_upper_percentile_da = xr.DataArray(
	list(unforced_trend_SEP_upper_percentile.values()),
	coords={'ICV_trend_year_upper': icv_years},
	dims=['ICV_trend_year_upper'],
)
    

In [ ]:
unforced_trend_SEP_lower_percentile_da

In [ ]:
# save the percentile data
dir_out = '/work/mh0033/m301036/OBS_LPS_revision/docs/data/FIG5/data/percentile/'
# use variable names that do not conflict with coordinate names
unforced_trend_SEP_lower_percentile_da.to_dataset(name="ICV_trend_lower").to_netcdf(dir_out+'internal_SEP_trend_lower_percentile.nc')
unforced_trend_SEP_upper_percentile_da.to_dataset(name="ICV_trend_upper").to_netcdf(dir_out+'internal_SEP_trend_upper_percentile.nc')